环境设置

目前AI整的要pytorch和torchvision

In [1]:
import torch
from torchvision import datasets, transforms

# ① 原始 MNIST（GAN 用，像素 ∈ [0,1]）
tf_raw = transforms.ToTensor()
train_raw = datasets.MNIST('./data', train=True,  download=True, transform=tf_raw)
test_raw  = datasets.MNIST('./data', train=False, download=True, transform=tf_raw)

# ② 二值化 MNIST（FVSBN / NADE 用，像素 ∈ {0,1}）
class Binarize:
    """tau: 阈值；stochastic=True 用随机二值化（更贴合 Bernoulli 似然）"""
    def __init__(self, tau=0.5, stochastic=False):
        self.tau, self.stochastic = tau, stochastic

    def __call__(self, x):                 # x: (1,28,28) in [0,1]
        if self.stochastic:
            return torch.bernoulli(x)
        return (x > self.tau).float()

tf_bin = transforms.Compose([transforms.ToTensor(), Binarize(tau=0.5)])
train_bin = datasets.MNIST('./data', train=True,  download=True, transform=tf_bin)
test_bin  = datasets.MNIST('./data', train=False, download=True, transform=tf_bin)

loader_bin = torch.utils.data.DataLoader(train_bin, batch_size=128, shuffle=True, num_workers=2)
loader_raw = torch.utils.data.DataLoader(train_raw, batch_size=128, shuffle=True, num_workers=2)

# sanity check：前景像素占比应 ≈ 0.13
x, _ = next(iter(loader_bin))
print('foreground ratio =', x.mean().item())

100%|██████████| 9.91M/9.91M [00:03<00:00, 3.28MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 137kB/s]
100%|██████████| 1.65M/1.65M [00:01<00:00, 1.03MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 2.50MB/s]


RuntimeError: DataLoader worker (pid(s) 2400, 6696) exited unexpectedly

任务1

In [ ]:
import torch, torch.nn as nn, torch.nn.functional as F

class FVSBN(nn.Module):
    """Fully Visible Sigmoid Belief Network（= 一阶链式自回归 logistic 模型）
       p(x_j=1 | x_{<j}) = sigmoid(w_j^T x_{<j} + b_j)
    """
    def __init__(self, D=784):
        super().__init__()
        self.D = D
        self.W = nn.Parameter(torch.zeros(D, D))      # 只用下三角部分
        self.b = nn.Parameter(torch.zeros(D))
        # 严格下三角 mask：第 j 行只能看到 0..j-1，防止信息泄漏
        self.register_buffer('mask', torch.tril(torch.ones(D, D), diagonal=-1))

    def logits(self, x):                              # x: (B, D) ∈ {0,1}
        return F.linear(x, self.W * self.mask, self.b)   # (B, D) 每列 = 一个像素的 logit

    def loss(self, x):                                # 最大似然 == 平均 BCE
        return F.binary_cross_entropy_with_logits(self.logits(x), x)

    @torch.no_grad()
    def sample(self, n, device='cpu'):
        x = torch.zeros(n, self.D, device=device)
        for j in range(self.D):                       # 784 步串行祖先采样
            pj = torch.sigmoid(self.logits(x)[:, j])
            x[:, j] = torch.bernoulli(pj)
        return x

# 训练
model = FVSBN().to(DEVICE)
opt = torch.optim.Adam(model.parameters(), lr=1e-3)
for epoch in range(50):
    for x, _ in loader_bin:
        x = x.view(x.size(0), -1).to(DEVICE)
        loss = model.loss(x)
        opt.zero_grad(); loss.backward(); opt.step()
    print(epoch, loss.item())                         # 参考：收敛到 ~0.25-0.30

# 生成 10 张
samples = model.sample(10, DEVICE).view(-1, 28, 28)

任务2

In [ ]:
import torch, torch.nn as nn, torch.nn.functional as F

class NADE(nn.Module):
    """Neural Autoregressive Density Estimator（单隐层，mask 保证自回归）
       h        = sigmoid(W x + c)，  W_{k,i} = 0 if i >= a_k
       logits_j = V_j^T h + b_j，     V_{j,k} = 0 if a_k > j
    """
    def __init__(self, D=784, H=512, seed=0):
        super().__init__()
        g = torch.Generator().manual_seed(seed)
        self.D, self.H = D, H
        order = torch.arange(D)                          # 也可用随机顺序
        a = torch.randint(1, D, (H,), generator=g)       # 每个隐单元的“最大可见下标”
        self.register_buffer('a', a)
        self.register_buffer('maskW', (order[None, :] < a[:, None]).float())   # (H, D)
        self.register_buffer('maskV', (a[None, :] <= order[:, None]).float())  # (D, H)
        self.W = nn.Parameter(torch.randn(H, D, generator=g) / D ** 0.5)
        self.V = nn.Parameter(torch.randn(D, H, generator=g) / H ** 0.5)
        self.c = nn.Parameter(torch.zeros(H))
        self.b = nn.Parameter(torch.zeros(D))

    def forward(self, x):                                # x: (B, D)
        h = torch.sigmoid(F.linear(x, self.W * self.maskW, self.c))
        return F.linear(h, self.V * self.maskV, self.b)  # (B, D) 全部条件 logits

    def loss(self, x):
        return F.binary_cross_entropy_with_logits(self.forward(x), x)

    @torch.no_grad()
    def sample(self, n, device='cpu'):
        x = torch.zeros(n, self.D, device=device)
        for j in range(self.D):                          # 串行祖先采样，batch 内并行
            pj = torch.sigmoid(self.forward(x)[:, j])
            x[:, j] = torch.bernoulli(pj)
        return x

# ---- MADE 对照：只需把 mask 换成 degree mask ----
# m0 = arange(D); m1 = randint(0, D-1, (H,)); m2 = arange(D)
# W1 = W1 * (m0[None,:] <= m1[:,None]);  W2 = W2 * (m1[None,:] < m2[:,None])
# 生成时一次前向即得全部 p_j

任务3

In [ ]:
class Generator(nn.Module):                              # z(100) -> 28x28x1
    def __init__(self, z=100, ngf=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(z, 7*7*ngf*4), nn.BatchNorm1d(7*7*ngf*4), nn.ReLU(True),
            nn.Unflatten(1, (ngf*4, 7, 7)),
            nn.ConvTranspose2d(ngf*4, ngf*2, 4, 2, 1), nn.BatchNorm2d(ngf*2), nn.ReLU(True),  # 14x14
            nn.ConvTranspose2d(ngf*2, ngf, 4, 2, 1), nn.BatchNorm2d(ngf), nn.ReLU(True),      # 28x28
            nn.Conv2d(ngf, 1, 3, 1, 1), nn.Tanh(),
        )
    def forward(self, z): return self.net(z)

class Discriminator(nn.Module):
    def __init__(self, ndf=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(1, ndf, 4, 2, 1), nn.LeakyReLU(0.2, True),                            # 14
            nn.Conv2d(ndf, ndf*2, 4, 2, 1), nn.BatchNorm2d(ndf*2), nn.LeakyReLU(0.2, True),  # 7
            nn.Flatten(), nn.Linear(ndf*2*7*7, 1),                                          # logits
        )
    def forward(self, x): return self.net(x)

G, D = Generator().to(DEVICE), Discriminator().to(DEVICE)
optG = torch.optim.Adam(G.parameters(), lr=2e-4, betas=(0.5, 0.999))
optD = torch.optim.Adam(D.parameters(), lr=2e-4, betas=(0.5, 0.999))
bce = nn.BCEWithLogitsLoss()

for epoch in range(30):
    for x, _ in loader_raw:
        x = x.to(DEVICE) * 2 - 1                 # [0,1] -> [-1,1]
        z = torch.randn(x.size(0), 100, device=DEVICE)
        # --- D ---
        optD.zero_grad()
        real, fake = D(x), D(G(z).detach())
        lossD = bce(real, torch.full_like(real, 0.9)) + bce(fake, torch.zeros_like(fake))
        lossD.backward(); optD.step()
        # --- G（non-saturating）---
        optG.zero_grad()
        gen = D(G(z))
        lossG = bce(gen, torch.ones_like(gen))
        lossG.backward(); optG.step()
        # 记录 real.mean(), fake.mean(), lossD, lossG —— 用于判断模式崩溃
    if epoch % 5 == 0:
        save_grid(G, epoch)                      # 5x4 训练进度网格